# 03 — Benchmarki: czas i poprawność HE

> Ten notebook uruchamia skrypty z `src/bench/`, zapisuje wyniki do
> `src/bench/results/*.csv` i pokazuje wykresy do prezentacji.

**Wymagania:** uruchom Jupyter z katalogu głównego projektu (`medical-he-stats`)
albo upewnij się, że kernel widzi pakiet `src` (patrz komórka poniżej).

## Co robi ten notebook

1. Ustala katalog projektu (`pyproject.toml`).
2. Generuje syntetycznych pacjentów i mierzy czasy HE vs NumPy (`bench_time`).
3. Porównuje błąd względny HE vs plaintext dla kilku kolumn (`bench_correctness`).
4. Rysuje wykresy PNG (te same co `python -m src.bench.plots`).

**Uwaga:** pierwsze uruchomienie może trwać **kilka minut** (keygen CKKS,
wiele rozmiarów N). Zmniejsz listę `--sizes` w komórce z benchmarkiem czasu,
jeśli testujesz na słabszej maszynie.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise RuntimeError("Nie znaleziono pyproject.toml — otwórz notebook z folderu projektu.")


ROOT = project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

## 1) Benchmark czasu (`bench_time`)

Zmienna `SIZES` — edytuj według potrzeb (np. pełna seria do pracy:
`100 500 1000 5000 10000 50000`).

In [ ]:
import subprocess
import sys

# --- edytuj tutaj ---
SIZES = [100, 500, 1000, 2000]  # szybki demo; do raportu: więcej punktów
REPEAT = 2
BUCKETS = 20

out_timing = ROOT / "src" / "bench" / "results" / "timing.csv"
out_timing.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-m",
    "src.bench.bench_time",
    "--sizes",
    *[str(s) for s in SIZES],
    "--repeat",
    str(REPEAT),
    "--buckets",
    str(BUCKETS),
    "--out",
    str(out_timing.relative_to(ROOT)),
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)

In [ ]:
import pandas as pd

timing = pd.read_csv(out_timing)
pd.options.display.float_format = "{:.2f}".format
display(timing)

## 2) Benchmark poprawności (`bench_correctness`)

Porównuje mean, variance, std (z HE jako sqrt(var)), medianę z histogramu — z oracle NumPy.

In [ ]:
import subprocess
import sys

out_correct = ROOT / "src" / "bench" / "results" / "correctness.csv"

cmd = [
    sys.executable,
    "-m",
    "src.bench.bench_correctness",
    "--n",
    "800",
    "--out",
    str(out_correct.relative_to(ROOT)),
    "--buckets",
    "30",
]
subprocess.run(cmd, cwd=str(ROOT), check=True)

corr = pd.read_csv(out_correct)
display(corr.groupby("stat")["rel_error"].agg(["max", "mean"]))

## 3) Wykresy (matplotlib)

Te same funkcje co `python -m src.bench.plots` — pliki trafiają do `src/bench/results/`.

In [ ]:
from IPython.display import Image, display

from src.bench.plots import plot_correctness_box, plot_he_vs_plain_bars, plot_time_scaling

OUT = ROOT / "src" / "bench" / "results"
OUT.mkdir(parents=True, exist_ok=True)

if out_timing.exists():
    plot_time_scaling(out_timing, OUT / "time_scaling.png")
    plot_he_vs_plain_bars(out_timing, OUT / "he_vs_plain.png")
    display(Image(filename=str(OUT / "time_scaling.png")))
    display(Image(filename=str(OUT / "he_vs_plain.png")))
else:
    print("Brak timing.csv — uruchom najpierw sekcję 1.")

if out_correct.exists():
    plot_correctness_box(out_correct, OUT / "correctness.png")
    display(Image(filename=str(OUT / "correctness.png")))
else:
    print("Brak correctness.csv — uruchom najpierw sekcję 2.")

## 4) Wiersz poleceń (bez notebooka)

```powershell
uv run python -m src.bench.bench_time --sizes 100 1000 10000
uv run python -m src.bench.bench_correctness --n 2000
uv run python -m src.bench.plots
```

Wykresy: `src/bench/results/time_scaling.png`, `he_vs_plain.png`, `correctness.png`.